# 01 — Reward Model Training

This notebook trains the reward model used by the PPO step. The base model is
`Qwen/Qwen2.5-1.5B-Instruct` with a scalar regression head and LoRA adapters; it is
fit on a 10 k subsample of `Anthropic/hh-rlhf` using the Bradley–Terry objective via
TRL's `RewardTrainer`.

**Output:** `outputs/reward_model/` containing the LoRA adapter, tokenizer and
training logs. The PPO notebook (`02_ppo_training.ipynb`) consumes this directory.

A T4 (16 GB) is enough thanks to 4-bit quantization; the training takes ~45 min.

## 1. Environment

Uncomment the install line the first time you run on a fresh machine.

In [ ]:
# !pip install -q -r ../requirements.txt

In [ ]:
import os, sys, json
from pathlib import Path

# Make `src/` importable regardless of where the notebook is launched from.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('Working dir:', Path.cwd())

In [ ]:
import torch
from transformers import TrainingArguments
from trl import RewardConfig, RewardTrainer

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.data.preferences import load_hh_rlhf_for_reward_model
from src.models.reward import build_reward_model

assert torch.cuda.is_available(), 'A GPU is required to train the reward model.'

cfg = load_config('configs/config.yaml')
seed_everything(cfg['seed'])
print(json.dumps(cfg['reward_model'], indent=2))

## 2. Load the preference data

We sub-sample HH-RLHF down to 10 k training pairs and 500 eval pairs. The loader
splits each transcript into `(prompt, chosen, rejected)` triples.

In [ ]:
train_ds, eval_ds = load_hh_rlhf_for_reward_model(
    num_train=cfg['reward_model']['num_train_samples'],
    num_eval=cfg['reward_model']['num_eval_samples'],
    seed=cfg['seed'],
)
print(f'train: {len(train_ds)}  eval: {len(eval_ds)}')
print('--- sample ---')
print('CHOSEN:', train_ds[0]['chosen'][:500], '...')
print('REJECTED:', train_ds[0]['rejected'][:500], '...')

## 3. Build the reward model

Qwen2.5-1.5B + scalar regression head, wrapped in LoRA adapters, with 4-bit
quantization on the frozen backbone.

In [ ]:
model, tokenizer = build_reward_model(
    model_name=cfg['base_model'],
    lora_cfg=cfg['lora'],
    quant_cfg=cfg['quantization'],
)
model.print_trainable_parameters()

## 4. Train

TRL's `RewardTrainer` handles tokenization of the `chosen`/`rejected` pairs and the
Bradley–Terry loss (`-log σ(r_chosen − r_rejected)`) for us. We tokenise both
responses to `max_length` and let the trainer mask padding.

In [ ]:
rm_cfg = cfg['reward_model']
output_dir = cfg['paths']['reward_model_dir']
Path(output_dir).mkdir(parents=True, exist_ok=True)

reward_config = RewardConfig(
    output_dir=output_dir,
    num_train_epochs=rm_cfg['num_train_epochs'],
    per_device_train_batch_size=rm_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size=rm_cfg['per_device_train_batch_size'],
    gradient_accumulation_steps=rm_cfg['gradient_accumulation_steps'],
    learning_rate=rm_cfg['learning_rate'],
    warmup_ratio=rm_cfg['warmup_ratio'],
    logging_steps=rm_cfg['logging_steps'],
    eval_strategy='steps',
    eval_steps=rm_cfg['eval_steps'],
    save_steps=rm_cfg['save_steps'],
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    max_length=rm_cfg['max_length'],
    report_to='none',
    seed=cfg['seed'],
)

trainer = RewardTrainer(
    model=model,
    args=reward_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

train_result = trainer.train()
print(train_result.metrics)

## 5. Save adapter + tokenizer

We save only the LoRA adapter; the next notebook will load the base model from the
Hub and attach this adapter.

In [ ]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

metrics = trainer.evaluate()
with open(Path(output_dir) / 'final_metrics.json', 'w') as fh:
    json.dump(metrics, fh, indent=2)
print('Saved to', output_dir)
print('Eval metrics:', metrics)

## 6. Sanity check — does the reward model prefer `chosen` over `rejected`?

On a small held-out batch we expect the *margin* (r_chosen − r_rejected) to be
positive most of the time. This is just a quick eyeball test; the test accuracy
computed by `RewardTrainer` above is the headline number.

In [ ]:
import numpy as np

model.eval()
n_check = 20
margins = []
with torch.no_grad():
    for row in eval_ds.select(range(min(n_check, len(eval_ds)))):
        toks_c = tokenizer(row['chosen'], truncation=True, max_length=rm_cfg['max_length'], return_tensors='pt').to(model.device)
        toks_r = tokenizer(row['rejected'], truncation=True, max_length=rm_cfg['max_length'], return_tensors='pt').to(model.device)
        r_c = model(**toks_c).logits.squeeze().item()
        r_r = model(**toks_r).logits.squeeze().item()
        margins.append(r_c - r_r)

margins = np.array(margins)
print(f'mean margin: {margins.mean():+.3f}  | sign agreement: {(margins > 0).mean():.1%}')